# Square Attack (Black-Box)

## Attacco Black-Box Score-Based su ResNet18

Square Attack è un attacco black-box che non usa i gradienti del modello target.
Lavora solo con le query al modello (accesso al vettore dei logit/score) e genera perturbazioni a forma di quadrati casuali.

In questo notebook:
1. Si carica ResNet18 come modello target
2. Si configura Square Attack tramite `torchattacks`
3. Si testa su vari epsilon
4. Si confrontano i risultati con gli attacchi white-box precedenti (tabella finale)

## 1. Setup e Import

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

import torchattacks

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path('..')
BATCH_SIZE = 32
IMG_SIZE = 224
N_CLASSES = 5

# ResNet18 — modello target (già addestrato)
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(in_features=512, out_features=N_CLASSES)
model = model.to(device)
model.load_state_dict(torch.load(ROOT / "models" / "best_resnet18.pt"))
model.eval()

NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

dataset_path = ROOT / "data" / "fruits-classification-stratified"

train_dataset = ImageFolder(root=dataset_path / 'train', transform=TRANSFORMS)
val_dataset   = ImageFolder(root=dataset_path / 'valid', transform=TRANSFORMS)
test_dataset  = ImageFolder(root=dataset_path / 'test',  transform=TRANSFORMS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

class_names = test_dataset.classes

results_path = ROOT / "results"
results_path.mkdir(exist_ok=True)

print(f"Device: {device}")
print(f"Classi: {class_names}")
print("Setup completato")


## 2. Square Attack — Teoria

Square Attack (Andriushchenko et al., 2020) è un attacco black-box **score-based**:
- Non richiede i gradienti del modello target
- Ad ogni iterazione, propone perturbazioni a forma di **quadrati casuali** in posizioni casuali dell'immagine
- Accetta la perturbazione solo se aumenta la loss (maximizing the loss = fool the model)
- Continua finché la predizione cambia oppure viene raggiunto il budget massimo di query

**Differenza fondamentale rispetto a FGSM/PGD/DeepFool:**

| Proprietà | White-Box | Square Attack |
|-----------|-----------|--------------|
| Accesso ai gradienti | Sì | No |
| Tipo di accesso | Gradient-based | Score-based (solo logit) |
| Query necessarie | 1 forward + 1 backward | N forward pass (budget) |
| Setting | White-box | Black-box |

**Implementazione:** si usa `torchattacks.Square` che gestisce la norma Linf.
Poiché il modello usa NORMALIZE internamente (immagini in `[0,1]` in input), si passa la normalizzazione a `torchattacks` tramite `set_normalization_used(mean, std)`.

## 3. Test Square Attack con epsilon diversi

Stessi epsilon usati nel transfer attack per confronto diretto.
Budget di query: `n_queries=1000` (valore standard per test principali).

Metriche:
- **Clean accuracy**: % immagini originali corrette
- **Adversarial accuracy**: % immagini perturbate ancora corrette
- **Accuracy drop**: differenza
- **ASR**: % immagini corrette prima e sbagliate dopo
- **Mean L2 / Mean Linf**: grandezza media delle perturbazioni
- **Tempo**: tempo totale per il test set

In [ ]:
epsilons_square = [0.001, 0.003, 0.005, 0.01]
n_queries       = 1000

# mean e std usate in NORMALIZE — servono a torchattacks per gestire il preprocessing
normalize_mean = [0.485, 0.456, 0.406]
normalize_std  = [0.229, 0.224, 0.225]

square_results = []
examples_store = {'success': [], 'failure': []}

model.eval()

for epsilon in epsilons_square:
    print(f"\nSquare Attack — epsilon={epsilon}, n_queries={n_queries}")

    # crea l'attacco: il modello riceve immagini in [0,1] e normalizza internamente
    # set_normalization_used dice a torchattacks come preprocessare prima di chiamare il modello
    atk = torchattacks.Square(
        model,
        norm='Linf',
        eps=epsilon,
        n_queries=n_queries,
        n_restarts=1,
        p_init=0.8,
        loss='margin',
        resc_schedule=True,
        seed=RANDOM_SEED,
        verbose=False
    )
    atk.set_normalization_used(mean=normalize_mean, std=normalize_std)

    correct_original = 0
    correct_adv      = 0
    attack_success   = 0
    total            = 0

    l2_list   = []
    linf_list = []

    start_time = time.time()

    for images, labels in tqdm(test_loader, desc=f"Square eps={epsilon}"):
        images, labels = images.to(device), labels.to(device)

        # predizioni su immagini originali
        with torch.no_grad():
            preds_original = torch.argmax(model(NORMALIZE(images)), dim=1)
            correct_original += (preds_original == labels).sum().item()

        # attacco Square — genera adv images
        adv_images = atk(images, labels)

        # predizioni su immagini adversarial
        with torch.no_grad():
            preds_adv = torch.argmax(model(NORMALIZE(adv_images)), dim=1)
            correct_adv += (preds_adv == labels).sum().item()

        success = (preds_original == labels) & (preds_adv != labels)
        attack_success += success.sum().item()
        total          += labels.size(0)

        perturbation = adv_images - images
        l2_norms   = perturbation.reshape(perturbation.size(0), -1).norm(p=2,   dim=1)
        linf_norms = perturbation.reshape(perturbation.size(0), -1).norm(p=float('inf'), dim=1)
        l2_list.extend(l2_norms.detach().cpu().numpy())
        linf_list.extend(linf_norms.detach().cpu().numpy())

        # raccoglie esempi per il plot
        for i in range(images.size(0)):
            pred_orig = preds_original[i].item()
            pred_adv  = preds_adv[i].item()
            true      = labels[i].item()
            ex = {
                'image':        images[i:i+1].detach().cpu(),
                'adv_image':    adv_images[i:i+1].detach().cpu(),
                'perturbation': perturbation[i:i+1].detach().cpu(),
                'pred_original': pred_orig,
                'pred_adv':      pred_adv,
                'true_label':    true,
                'l2_norm':       l2_norms[i].item(),
            }
            if pred_orig == true and pred_adv != true:
                examples_store['success'].append(ex)
            elif pred_orig == true and pred_adv == true:
                examples_store['failure'].append(ex)

    elapsed = time.time() - start_time

    accuracy     = correct_original / total
    accuracy_adv = correct_adv      / total
    accuracy_drop = accuracy - accuracy_adv
    asr          = attack_success / correct_original if correct_original > 0 else 0
    mean_l2      = float(np.mean(l2_list))
    mean_linf    = float(np.mean(linf_list))

    print(f"  Accuracy (clean):      {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Accuracy (adversarial):{accuracy_adv:.4f} ({accuracy_adv*100:.2f}%)")
    print(f"  Accuracy Drop:         {accuracy_drop:.4f} ({accuracy_drop*100:.2f}%)")
    print(f"  ASR:                   {asr:.4f} ({asr*100:.2f}%)")
    print(f"  Mean L2:               {mean_l2:.4f}")
    print(f"  Mean Linf:             {mean_linf:.4f}")
    print(f"  Tempo:                 {elapsed:.2f}s")

    square_results.append({
        'epsilon':      epsilon,
        'n_queries':    n_queries,
        'accuracy':     accuracy,
        'accuracy_adv': accuracy_adv,
        'accuracy_drop':accuracy_drop,
        'asr':          asr,
        'mean_l2':      mean_l2,
        'mean_linf':    mean_linf,
        'time_s':       elapsed,
    })

print("\nSquare Attack completato per tutti gli epsilon")


## 4. Plot immagini perturbate

Stesso stile 3×4 degli altri notebook: originale / perturbazione / adversarial.
Si mostrano 2 attacchi riusciti e 2 falliti (immagini con epsilon=0.01).

In [ ]:
selected = examples_store['success'][:2] + examples_store['failure'][:2]

if len(selected) < 4:
    print(f"Esempi insufficienti per il plot ({len(selected)}/4)")
else:
    fig, axes = plt.subplots(3, 4, figsize=(15, 9))

    for img_idx in range(4):
        ex = selected[img_idx]

        img_orig = np.transpose(np.clip(ex['image'][0].numpy(), 0, 1), (1, 2, 0))

        pert = ex['perturbation'][0].numpy()
        pert = np.transpose(pert, (1, 2, 0))
        pert_magnitude = np.mean(np.abs(pert), axis=2)

        img_adv = np.transpose(np.clip(ex['adv_image'][0].numpy(), 0, 1), (1, 2, 0))

        true_label   = ex['true_label']
        pred_original = ex['pred_original']
        pred_adv      = ex['pred_adv']
        l2_norm       = ex['l2_norm']

        ax = axes[0, img_idx]
        ax.imshow(img_orig)
        ax.set_title(
            f"Originale\nTrue: {class_names[true_label]}\nPred: {class_names[pred_original]}",
            fontsize=10
        )
        ax.axis('off')

        ax = axes[1, img_idx]
        ax.imshow(pert_magnitude, cmap='gray')
        ax.set_title(f"Perturbazione\nL2={l2_norm:.4f}", fontsize=10)
        ax.axis('off')

        ax = axes[2, img_idx]
        ax.imshow(img_adv)
        color = 'red' if pred_adv != true_label else 'green'
        ax.set_title(
            f"Adversarial\nTrue: {class_names[true_label]}\nPred: {class_names[pred_adv]}",
            fontsize=10,
            color=color
        )
        ax.axis('off')

    plt.suptitle("Square Attack — esempi (epsilon=0.01, n_queries=1000)", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_path / "Square_attack_examples.png", dpi=120, bbox_inches='tight')
    plt.show()
    print("Salvato: Square_attack_examples.png")


## 5. Plot metriche Square Attack

Stessi 4 grafici degli altri notebook: accuracy clean vs adv, accuracy drop, ASR, tempo.

In [ ]:
square_df = pd.DataFrame(square_results)

eps        = square_df['epsilon'].tolist()
acc        = square_df['accuracy'].tolist()
acc_adv    = square_df['accuracy_adv'].tolist()
acc_drop   = square_df['accuracy_drop'].tolist()
asr        = square_df['asr'].tolist()
times      = square_df['time_s'].tolist()
eps_labels = [str(e) for e in eps]

sns.set(style='whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

ax = axes[0, 0]
ax.plot(eps, acc,     marker='o', label='Accuracy (clean)',      linewidth=2, markersize=8)
ax.plot(eps, acc_adv, marker='s', label='Accuracy (adversarial)',linewidth=2, markersize=8)
ax.set_xlabel('epsilon', fontsize=11)
ax.set_xticks(eps)
ax.set_xticklabels(eps_labels)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy: clean vs adversarial', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
sns.barplot(x=eps_labels, y=acc_drop, palette='Reds', ax=ax)
ax.set_xlabel('epsilon', fontsize=11)
ax.set_ylabel('Accuracy drop', fontsize=11)
ax.set_title('Accuracy Drop per epsilon', fontsize=12, fontweight='bold')

ax = axes[1, 0]
sns.barplot(x=eps_labels, y=asr, palette='Blues', ax=ax)
ax.set_xlabel('epsilon', fontsize=11)
ax.set_ylabel('ASR', fontsize=11)
ax.set_title('Attack Success Rate per epsilon', fontsize=12, fontweight='bold')

ax = axes[1, 1]
sns.barplot(x=eps_labels, y=times, palette='Greens', ax=ax)
ax.set_xlabel('epsilon', fontsize=11)
ax.set_ylabel('Time (s)', fontsize=11)
ax.set_title("Tempo di esecuzione per epsilon", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(results_path / 'Square_metrics_summary.png', dpi=120, bbox_inches='tight')
plt.show()

# salva CSV
square_df.to_csv(results_path / 'Square_metrics_summary.csv', index=False)
print("Salvati: Square_metrics_summary.png e Square_metrics_summary.csv")
print(square_df.to_string(index=False))


## 6. Confronto finale — White-Box vs Black-Box

Tabella unificata che mette insieme tutti gli attacchi del progetto:
- FGSM, PGD, DeepFool (white-box, da notebook 02-04)
- Transfer FGSM/PGD/DeepFool (black-box transfer, da notebook 05)
- Square Attack (black-box score-based, questo notebook)

Carica i CSV già salvati e costruisce la tabella finale.

In [ ]:
final_rows = []

# ---- White-box: FGSM ----
fgsm_wb = pd.read_csv(results_path / "FGSM_metrics_summary.csv")
for _, row in fgsm_wb.iterrows():
    final_rows.append({
        'Attack':          'FGSM',
        'Setting':         'white-box',
        'Target model':    'ResNet18',
        'Main parameter':  f"eps={row['epsilon']}",
        'Clean acc':       row['accuracy'],
        'Adv acc':         row['accuracy_adv'],
        'Drop':            row['accuracy_drop'],
        'ASR':             row['asr'],
    })

# ---- White-box: PGD ----
pgd_wb = pd.read_csv(results_path / "PGD_metrics_summary.csv")
for _, row in pgd_wb.iterrows():
    final_rows.append({
        'Attack':          'PGD',
        'Setting':         'white-box',
        'Target model':    'ResNet18',
        'Main parameter':  f"eps={row['epsilon']}",
        'Clean acc':       row['accuracy'],
        'Adv acc':         row['accuracy_adv'],
        'Drop':            row['accuracy_drop'],
        'ASR':             row['asr'],
    })

# ---- White-box: DeepFool ----
df_wb = pd.read_csv(results_path / "DF_metrics_summary.csv")
for _, row in df_wb.iterrows():
    final_rows.append({
        'Attack':          'DeepFool',
        'Setting':         'white-box',
        'Target model':    'ResNet18',
        'Main parameter':  f"overshoot={row['overshoot']}",
        'Clean acc':       row['accuracy'],
        'Adv acc':         row['accuracy_adv'],
        'Drop':            row['accuracy_drop'],
        'ASR':             row['asr'],
    })

# ---- Black-box Transfer: FGSM, PGD, DeepFool ----
transfer_full = pd.read_csv(results_path / "Transfer_original_vs_adv.csv")
for _, row in transfer_full.iterrows():
    attack = row['attack']
    param  = f"eps={row['epsilon']}" if attack != 'DeepFool' else f"overshoot={row['epsilon']}"
    final_rows.append({
        'Attack':          attack,
        'Setting':         'black-box (transfer)',
        'Target model':    'MobileNetV2',
        'Main parameter':  f"{param}, source=ResNet18",
        'Clean acc':       row['target_clean_acc'],
        'Adv acc':         row['target_adv_acc'],
        'Drop':            row['target_clean_acc'] - row['target_adv_acc'],
        'ASR':             row['transfer_asr'],
    })

# ---- Black-box: Square Attack ----
for _, row in square_df.iterrows():
    final_rows.append({
        'Attack':          'Square',
        'Setting':         'black-box (score-based)',
        'Target model':    'ResNet18',
        'Main parameter':  f"eps={row['epsilon']}, q={int(row['n_queries'])}",
        'Clean acc':       row['accuracy'],
        'Adv acc':         row['accuracy_adv'],
        'Drop':            row['accuracy_drop'],
        'ASR':             row['asr'],
    })

final_df = pd.DataFrame(final_rows)

# formatta le colonne numeriche
for col in ['Clean acc', 'Adv acc', 'Drop', 'ASR']:
    final_df[col] = final_df[col].map(lambda x: f"{x:.4f}")

print("Tabella confronto finale — White-Box vs Black-Box:")
print(final_df.to_string(index=False))

final_df.to_csv(results_path / "Final_comparison.csv", index=False)
print("\nSalvato: Final_comparison.csv")


In [ ]:
# ---- Bar chart finale: ASR per attacco e setting ----
final_df_num = pd.DataFrame(final_rows)  # versione con valori numerici

sns.set(style='whitegrid')

fig, ax = plt.subplots(figsize=(16, 6))

attack_labels = [
    f"{row['Attack']}\n{row['Setting']}\n{row['Main parameter']}"
    for _, row in final_df_num.iterrows()
]

colors = []
palette = {
    'white-box':             'steelblue',
    'black-box (transfer)':  'coral',
    'black-box (score-based)':'mediumseagreen',
}
for _, row in final_df_num.iterrows():
    colors.append(palette[row['Setting']])

bars = ax.bar(range(len(final_df_num)), final_df_num['ASR'], color=colors)

ax.set_xticks(range(len(final_df_num)))
ax.set_xticklabels(attack_labels, fontsize=7, rotation=45, ha='right')
ax.set_ylabel('ASR (Attack Success Rate)', fontsize=11)
ax.set_title('Confronto ASR — White-Box vs Black-Box', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)

# legenda manuale
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue',      label='White-box'),
    Patch(facecolor='coral',          label='Black-box (transfer)'),
    Patch(facecolor='mediumseagreen', label='Black-box (score-based)'),
]
ax.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.savefig(results_path / "Final_comparison_plot.png", dpi=120, bbox_inches='tight')
plt.show()
print("Salvato: Final_comparison_plot.png")
